# Host Qwen3.5-9B trên Google Colab và gọi từ local

Notebook độc lập: tải **Qwen/Qwen3.5-9B từ Hugging Face**, chạy inference trên GPU Colab,
phục vụ API chat văn bản bằng FastAPI, rồi mở URL HTTPS bằng ngrok.
Model chạy trên **Colab**, Hugging Face cung cấp weights; notebook không tạo HF Space hay Inference Endpoint.

**Chuẩn bị**

1. Upload notebook vào [Google Colab](https://colab.research.google.com/), chọn
   **Runtime → Change runtime type → GPU**. Ưu tiên L4/A100; chế độ 4-bit được thiết kế
   để thử trên T4 16 GB, nhưng chưa có phép đo thực tế trong môi trường này.
2. Tạo tài khoản [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken).
   Trong Colab **Secrets** (biểu tượng chìa khóa), tạo `NGROK_AUTHTOKEN` và
   `QWEN_API_KEY` (tự chọn chuỗi bí mật ASCII ít nhất 16 ký tự), bật Notebook access.
   Model công khai được tải trực tiếp, không cần Hugging Face token.
   Không lưu key trong code/output.
3. Chạy từng cell theo thứ tự. Cell cài đặt chỉ chạy khi bạn đặt `INSTALL_PACKAGES = True`.
   Sau khi cài, **restart session**, đặt lại `INSTALL_PACKAGES = False`, chạy tiếp.
4. Sau khi smoke test thành công, đặt `OPEN_PUBLIC_TUNNEL = True` để mở URL local có thể gọi.
   ngrok có thể tải binary khi mở tunnel lần đầu; token được truyền trong bộ nhớ.
5. Dùng file `test_local.py` đi kèm trên máy local; xem ví dụ ở cuối notebook.

API có `/health`, `/v1/models`, `/v1/chat/completions`, yêu cầu Bearer key.
Đây là tập con API chat: **text-only, không streaming, không tool calling**.
Mặc định tắt thinking, context tổng 4096 token, xử lý một request sinh văn bản mỗi lần.
Weights gốc vẫn phải tải trước khi quantize; cần khoảng 30 GB disk trống.
Colab ngắt phiên thì API dừng; chạy lại tunnel có thể đổi URL.

In [ ]:
import importlib.metadata
import importlib.util
import shlex
import subprocess
import sys

INSTALL_PACKAGES = False  # Set True only to approve installation in this Colab runtime.
PACKAGES = [
    "transformers==5.13.1",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.49,<1",
    "fastapi>=0.115,<1",
    "uvicorn>=0.30,<1",
    "pyngrok>=7.2,<9",
]
install_command = [sys.executable, "-m", "pip", "install", *PACKAGES]
print("Proposed Colab command:", shlex.join(install_command))
if INSTALL_PACKAGES:
    if importlib.util.find_spec("google.colab") is None:
        raise RuntimeError("This installation cell is intended for Google Colab only.")
    subprocess.run(install_command, check=True)
    print("Installation finished. Restart the session before continuing.")
else:
    print("No packages installed. Review the command and enable INSTALL_PACKAGES if needed.")


## Kiểm tra runtime và lấy secret

Cell tiếp theo dừng nếu thiếu dependency hoặc CUDA; không tự cài và không chuyển sang CPU.
PyTorch dùng sẵn bản CUDA của Colab. Không cần cài FlashAttention, causal-conv1d hay flash-linear-attention;
thiếu kernel tối ưu thì Transformers dùng phép toán PyTorch chậm hơn và có thể tốn thêm VRAM.
Nếu cảnh báo liên quan kernel xuất hiện, đó không phải bằng chứng inference đã thành công.

In [ ]:
from getpass import getpass
from pathlib import Path
import shutil

for module in ("torch", "transformers", "accelerate", "bitsandbytes", "fastapi", "uvicorn", "pyngrok"):
    if importlib.util.find_spec(module) is None:
        raise ModuleNotFoundError(
            f"No module named '{module}'. Review and approve the install cell, then restart."
        )

import torch
import transformers

if not hasattr(transformers, "Qwen3_5ForConditionalGeneration"):
    raise RuntimeError("Transformers lacks Qwen3.5 support. Review the install cell and restart.")
print("Python:", sys.version.split()[0])
for package in ("torch", "transformers", "accelerate", "bitsandbytes", "fastapi", "uvicorn", "pyngrok"):
    print(f"{package}: {importlib.metadata.version(package)}")
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime. CPU fallback is disabled.")

MODEL_ID = "Qwen/Qwen3.5-9B"
MODEL_REVISION = "main"  # Resolved to one immutable commit before loading tokenizer and model.
LOAD_IN_4BIT = True
MAX_CONTEXT_TOKENS = 4096  # Includes both formatted prompt and generated output.
PORT = 8000

free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM free/total: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
print("Compute dtype:", compute_dtype)
if free_bytes < 12 * 2**30:
    raise RuntimeError("Less than 12 GiB VRAM free. Restart with a larger/free GPU.")
if not LOAD_IN_4BIT and free_bytes < 24 * 2**30:
    raise RuntimeError("Use 4-bit mode or a GPU with at least 24 GiB free for full precision.")
print(f"Disk free: {shutil.disk_usage(Path.cwd()).free / 2**30:.1f} GiB")
if shutil.disk_usage(Path.cwd()).free < 30 * 2**30:
    print("Low disk space: uncached original model weights may not fit.")


def read_secret(name: str, required: bool = True) -> str | None:
    """Read a Colab secret or prompt without echoing its value."""
    from google.colab import userdata
    try:
        value = userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        value = None
    if not value and required:
        value = getpass(f"{name}: ")
    return value or None


API_KEY = read_secret("QWEN_API_KEY")
if not API_KEY or len(API_KEY) < 16 or not API_KEY.isascii() or any(char.isspace() for char in API_KEY):
    raise ValueError("Use an ASCII API key with at least 16 characters and no whitespace.")
print("Secrets loaded; values are hidden.")


## Tải model một lần

Cell này dùng lớp `Qwen3_5ForConditionalGeneration` đúng với checkpoint gốc,
nhưng API bên dưới chỉ nhận văn bản. `device_map={"": 0}` yêu cầu toàn bộ model ở GPU 0;
nếu không đủ VRAM thì báo lỗi. Quantization NF4 có thể làm thay đổi câu trả lời so với weights gốc.
T4 dùng FP16, GPU hỗ trợ BF16 sẽ dùng BF16. Chưa kiểm chứng tốc độ hay mức VRAM thực tế trên Colab.

In [ ]:
from typing import Any


class QwenEngine:
    """Keep one CUDA model loaded and generate text within a bounded context."""

    def __init__(
        self, model_id: str, revision: str,
        load_in_4bit: bool, max_context: int,
    ) -> None:
        import torch
        from huggingface_hub import HfApi
        from transformers import AutoTokenizer, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is required; no CPU fallback.")
        self.torch = torch
        self.model_id = model_id
        self.max_context = max_context
        self.revision = HfApi(token=False).model_info(model_id, revision=revision).sha
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, revision=self.revision, token=False, trust_remote_code=False,
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        options = {
            "revision": self.revision,
            "token": False,
            "trust_remote_code": False,
            "device_map": {"": 0},
            "dtype": dtype,
            "attn_implementation": "sdpa",
        }
        if load_in_4bit:
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=dtype,
            )
        self.model = Qwen3_5ForConditionalGeneration.from_pretrained(model_id, **options)
        self.model.eval()
        self.device = self.model.get_input_embeddings().weight.device
        if self.device.type != "cuda":
            raise RuntimeError("Model input embeddings are not on CUDA.")
        print("Loaded model:", self.model_id)
        print("Resolved revision:", self.revision)
        print(f"Model footprint: {self.model.get_memory_footprint() / 2**30:.2f} GiB")

    def complete(
        self, messages: list[dict[str, str]], max_tokens: int,
        temperature: float, top_p: float, enable_thinking: bool,
    ) -> dict[str, Any]:
        """Reject oversized prompts instead of silently truncating history."""
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
        inputs = self.tokenizer(text, return_tensors="pt", add_special_tokens=False)
        prompt_tokens = inputs["input_ids"].shape[-1]
        if prompt_tokens + max_tokens > self.max_context:
            raise ValueError(
                f"Prompt ({prompt_tokens}) + max_tokens ({max_tokens}) exceeds "
                f"context budget ({self.max_context}). Shorten the messages/output."
            )
        settings = {"do_sample": temperature > 0}
        if temperature > 0:
            settings.update(temperature=temperature, top_p=top_p, top_k=20)
        else:
            settings.update(temperature=None, top_p=None, top_k=None)
        try:
            inputs = inputs.to(self.device)
            with self.torch.inference_mode():
                output = self.model.generate(
                    **inputs, **settings, max_new_tokens=max_tokens,
                    pad_token_id=self.tokenizer.pad_token_id,
                    use_cache=True, logits_to_keep=1,
                )
            generated = output[0, prompt_tokens:]
            content = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
            completion_tokens = len(generated)
            eos_ids = self.model.generation_config.eos_token_id
            eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
            ended = bool(completion_tokens and generated[-1].item() in eos_ids)
        except self.torch.cuda.OutOfMemoryError as exc:
            raise MemoryError("CUDA ran out of memory during generation.") from exc
        return {
            "content": content,
            "finish_reason": "stop" if ended or completion_tokens < max_tokens else "length",
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
        }


In [ ]:
if globals().get("engine") is not None:
    raise RuntimeError("A model is already loaded. Restart the session before loading again.")
engine = QwenEngine(MODEL_ID, MODEL_REVISION, LOAD_IN_4BIT, MAX_CONTEXT_TOKENS)


## API có xác thực và kiểm soát request

Chỉ nhận nội dung văn bản; tối đa 32 message, 64.000 ký tự tổng cộng, tối đa 1024 token output.
`max_tokens` tính cả thinking nếu bật. `enable_thinking=true` trả nguyên văn phần thinking trong
`message.content`; API không tách `reasoning_content`. Context được kiểm tra sau khi thêm chat template.
Request sinh văn bản thứ hai khi GPU đang bận nhận HTTP 429, không xếp hàng vô hạn.

In [ ]:
import logging
import secrets
import threading
import time
import uuid
from typing import Literal

from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, ConfigDict, Field, model_validator


class ChatMessage(BaseModel):
    """A text-only message supported by this server."""

    model_config = ConfigDict(extra="forbid")
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=16000)


class ChatRequest(BaseModel):
    """The supported subset of chat-completion parameters."""

    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)
    model: str = "Qwen/Qwen3.5-9B"
    messages: list[ChatMessage] = Field(min_length=1, max_length=32)
    max_tokens: int = Field(default=256, ge=1, le=1024, strict=True)
    temperature: float = Field(default=0.7, ge=0, le=2)
    top_p: float = Field(default=0.8, gt=0, le=1)
    enable_thinking: bool = False
    stream: Literal[False] = False

    @model_validator(mode="after")
    def validate_messages(self):
        if sum(len(message.content) for message in self.messages) > 64000:
            raise ValueError("Total message content exceeds 64000 characters.")
        if not any(message.role == "user" for message in self.messages):
            raise ValueError("At least one user message is required.")
        if any(message.role == "system" for message in self.messages[1:]):
            raise ValueError("System message is only allowed at the beginning.")
        if any(not message.content.strip() for message in self.messages):
            raise ValueError("Messages must contain non-whitespace text.")
        return self


def create_app(engine, api_key: str) -> FastAPI:
    """Serve a loaded engine without exposing model errors to API clients."""
    if len(api_key) < 16 or not api_key.isascii() or any(char.isspace() for char in api_key):
        raise ValueError("Use an ASCII API key of at least 16 characters without whitespace.")
    bearer = HTTPBearer(auto_error=False)

    def authenticate(credentials: HTTPAuthorizationCredentials | None = Depends(bearer)) -> None:
        supplied = credentials.credentials if credentials else ""
        if not secrets.compare_digest(supplied.encode("utf-8"), api_key.encode("utf-8")):
            raise HTTPException(401, "Invalid or missing bearer key.", headers={"WWW-Authenticate": "Bearer"})

    app = FastAPI(title="Qwen Colab Text API", docs_url=None, redoc_url=None, openapi_url=None)
    app.state.generation_lock = threading.Lock()
    app.state.engine = engine

    @app.get("/health", dependencies=[Depends(authenticate)])
    def health() -> dict:
        return {
            "status": "ok", "model": engine.model_id, "revision": engine.revision,
            "max_context_tokens": engine.max_context,
            "busy": app.state.generation_lock.locked(),
        }

    @app.get("/v1/models", dependencies=[Depends(authenticate)])
    def list_models() -> dict:
        return {"object": "list", "data": [{
            "id": engine.model_id, "object": "model", "created": 0, "owned_by": "colab",
        }]}

    @app.post("/v1/chat/completions", dependencies=[Depends(authenticate)])
    def complete(request: ChatRequest) -> dict:
        if request.model != engine.model_id:
            raise HTTPException(404, "Unknown model. Use /v1/models.")
        if not app.state.generation_lock.acquire(blocking=False):
            raise HTTPException(429, "GPU is busy. Retry after the current request finishes.", headers={"Retry-After": "5"})
        try:
            result = engine.complete(
                [message.model_dump() for message in request.messages],
                request.max_tokens, request.temperature, request.top_p, request.enable_thinking,
            )
        except ValueError as exc:
            raise HTTPException(400, str(exc)) from exc
        except MemoryError as exc:
            raise HTTPException(503, "GPU out of memory. Reduce context/output or use a larger GPU.") from exc
        except Exception as exc:
            logging.getLogger("qwen_colab").exception("Generation failed")
            raise HTTPException(500, "Generation failed. Check the Colab runtime logs.") from exc
        finally:
            app.state.generation_lock.release()
        return {
            "id": f"chatcmpl-{uuid.uuid4().hex}",
            "object": "chat.completion", "created": int(time.time()), "model": engine.model_id,
            "choices": [{
                "index": 0, "message": {"role": "assistant", "content": result["content"]},
                "finish_reason": result["finish_reason"],
            }],
            "usage": result["usage"],
        }

    return app


In [ ]:
import json
from urllib.error import URLError
from urllib.request import Request, urlopen

import uvicorn

if globals().get("server_thread") is not None and server_thread.is_alive():
    raise RuntimeError("Server is already running. Use the stop cell before starting again.")
app = create_app(engine, API_KEY)
server = uvicorn.Server(uvicorn.Config(
    app, host="127.0.0.1", port=PORT, log_level="warning", access_log=False,
    loop="asyncio", http="h11", ws="none",
))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for attempt in range(150):
    if not server_thread.is_alive():
        raise RuntimeError("API server stopped during startup. Check port and runtime logs.")
    try:
        request = Request(f"http://127.0.0.1:{PORT}/health", headers={"Authorization": f"Bearer {API_KEY}"})
        with urlopen(request, timeout=1) as response:
            health_result = json.load(response)
        print(health_result)
        break
    except (URLError, TimeoutError):
        time.sleep(0.2)
else:
    server.should_exit = True
    raise RuntimeError("Server did not become ready within 30 seconds.")


In [ ]:
smoke_payload = {
    "model": MODEL_ID,
    "messages": [{"role": "user", "content": "Trả lời bằng một câu: thủ đô Việt Nam là gì?"}],
    "max_tokens": 64,
    "temperature": 0,
    "enable_thinking": False,
}
smoke_request = Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    data=json.dumps(smoke_payload).encode("utf-8"),
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
)
smoke_test_passed = False
with urlopen(smoke_request, timeout=600) as response:
    smoke_result = json.load(response)
if not smoke_result["choices"][0]["message"]["content"].strip():
    raise RuntimeError("The model returned no text. Check GPU/runtime before opening the tunnel.")
print(json.dumps(smoke_result, ensure_ascii=False, indent=2))
smoke_test_passed = True


## Mở HTTPS tunnel

Chỉ mở sau smoke test. Đặt `OPEN_PUBLIC_TUNNEL = True` là đồng ý khởi chạy ngrok
và tải ngrok binary nếu thiếu. Dùng `QWEN_API_KEY` riêng với `NGROK_AUTHTOKEN`.
URL được in ra; key không được in. Giữ phiên Colab đang hoạt động khi gọi từ local.

In [ ]:
OPEN_PUBLIC_TUNNEL = False

if OPEN_PUBLIC_TUNNEL:
    from pyngrok import conf, ngrok

    if not globals().get("smoke_test_passed", False) or not server_thread.is_alive():
        raise RuntimeError("Start the API and pass the model smoke test first.")
    if globals().get("public_url"):
        raise RuntimeError("A tunnel is already open. Use the stop cell before reopening.")
    ngrok_token = read_secret("NGROK_AUTHTOKEN")
    ngrok_config = conf.PyngrokConfig(auth_token=ngrok_token)
    tunnel = ngrok.connect(
        addr=f"127.0.0.1:{PORT}", proto="http", bind_tls=True,
        inspect=False, pyngrok_config=ngrok_config,
    )
    public_url = tunnel.public_url
    ngrok_token = None
    if not public_url.startswith("https://"):
        ngrok.disconnect(public_url, pyngrok_config=ngrok_config)
        public_url = None
        raise RuntimeError("Expected an HTTPS ngrok URL.")
    print("QWEN_BASE_URL:", public_url)
    print("API base URL:", public_url + "/v1")
else:
    print("Tunnel is disabled. Set OPEN_PUBLIC_TUNNEL = True when ready.")


## Gọi từ máy local

File `test_local.py` chỉ dùng Python standard library, không cần cài `requests` hay `openai`.
Dùng key giống `QWEN_API_KEY` trong Colab. Nếu không đặt biến môi trường, script hỏi key bằng getpass.

```bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate DL_Env
python /path/to/test_local.py --base-url "https://YOUR-NGROK-DOMAIN" \
  --prompt "Giải thích RAG bằng tiếng Việt trong 3 câu." --max-tokens 256
```

Có thể đặt `QWEN_BASE_URL`, `QWEN_API_KEY` trong environment. `--json` in toàn bộ response.
`--thinking --max-tokens 1024` bật thinking nhưng có thể hết token trước câu trả lời cuối;
output gồm thinking. API không hỗ trợ `stream=true`, ảnh/video, tools, response_format hay các tham số ngoài schema.

**Xử lý lỗi**

- `401`: key không khớp; dùng API key, không dùng ngrok authtoken.
- `429`: chờ request đang chạy xong.
- `400` context: giảm prompt/lịch sử hoặc output; không tự cắt nội dung.
- `503` OOM: giảm `MAX_CONTEXT_TOKENS`/`max_tokens`, khởi động lại session và tải model lại,
  hoặc chọn GPU lớn hơn. Không đổi model khi bạn chưa chủ động thay cấu hình.
- Timeout/URL lỗi: kiểm tra Colab còn kết nối và URL mới nhất; T4/reference kernels có thể chậm.
- Lỗi import sau cài đặt: restart session trước khi import lại; không thay PyTorch/CUDA một cách tự động.

Notebook chưa được chạy với weights thật trong phiên phát triển; smoke test phải thành công trên Colab
trước khi coi endpoint sẵn sàng.

## Dừng server (chỉ chạy khi muốn kết thúc)

Cell cuối để chế độ tắt mặc định để Run all không đóng API ngay sau khi mở.
Đặt `STOP_SERVER = True` rồi chạy cell này để đóng tunnel và server của notebook.
Model vẫn nằm trong GPU; **Disconnect and delete runtime** khi muốn giải phóng toàn bộ.

In [ ]:
STOP_SERVER = False

if STOP_SERVER:
    if globals().get("public_url"):
        ngrok.disconnect(public_url, pyngrok_config=ngrok_config)
        public_url = None
    if globals().get("server") is not None:
        server.should_exit = True
    if globals().get("server_thread") is not None:
        server_thread.join(timeout=10)
        if server_thread.is_alive():
            raise RuntimeError("Generation is still running. Wait and run this stop cell again.")
    smoke_test_passed = False
    print("Tunnel and API stopped. Model remains loaded until runtime disconnects.")
else:
    print("Server keeps running. Set STOP_SERVER = True only when finished.")


## Tài liệu nguồn

- [Model card Qwen/Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B)
- [Lớp model Qwen3.5 trong Transformers](https://huggingface.co/docs/transformers/model_doc/qwen3_5)
- [Bitsandbytes 4-bit/NF4](https://huggingface.co/docs/transformers/quantization/bitsandbytes)
- [pyngrok API](https://pyngrok.readthedocs.io/en/latest/api.html)

Cấu hình context ngắn ở đây phục vụ thử nghiệm trên một GPU; không khai thác context dài của model.